In [ ]:
import sqlite3

# 1. DB 파일 연결 (없으면 자동 생성, 깃에 공유 가능)
conn = sqlite3.connect("complaints_system.db")
cursor = conn.cursor()

# 외래키(Foreign Key) 활성화
cursor.execute("PRAGMA foreign_keys = ON;")

# 2. DDL 실행 (테이블 3개 생성)
# 2-1. users 테이블
cursor.execute(
    """
CREATE TABLE IF NOT EXISTS users (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    username TEXT UNIQUE NOT NULL,
    password TEXT NOT NULL,
    name TEXT NOT NULL,
    role TEXT NOT NULL CHECK(role IN ('STUDENT', 'ADMIN'))
);
"""
)

# 2-2. complaints 테이블
cursor.execute(
    """
CREATE TABLE IF NOT EXISTS complaints (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    user_id INTEGER NOT NULL,
    category TEXT NOT NULL,
    raw_content TEXT NOT NULL,
    summary TEXT NOT NULL,
    photo_url TEXT,
    status TEXT NOT NULL DEFAULT '접수',
    created_at DATETIME DEFAULT CURRENT_TIMESTAMP,
    updated_at DATETIME DEFAULT CURRENT_TIMESTAMP,
    FOREIGN KEY (user_id) REFERENCES users(id) ON DELETE CASCADE
);
"""
)

# 2-3. complaint_likes 테이블 (complaint_id + user_id 복합 PK)
cursor.execute(
    """
CREATE TABLE IF NOT EXISTS complaint_likes (
    complaint_id INTEGER NOT NULL,
    user_id INTEGER NOT NULL,
    created_at DATETIME DEFAULT CURRENT_TIMESTAMP,
    PRIMARY KEY (complaint_id, user_id),
    FOREIGN KEY (complaint_id) REFERENCES complaints(id) ON DELETE CASCADE,
    FOREIGN KEY (user_id) REFERENCES users(id) ON DELETE CASCADE
);
"""
)

# 3. Seed 데이터 주입 (기본 계정 2개)
cursor.execute(
    """
INSERT OR IGNORE INTO users (id, username, password, name, role)
VALUES 
    (1, 'student1', '1234', '이학생', 'STUDENT'),
    (2, 'admin1', '1234', '나관리', 'ADMIN');
"""
)

conn.commit()
conn.close()

print(
    "✅ DB 테이블 3개(users, complaints, complaint_likes) 및 Seed 데이터 생성 완료!"
)